In [0]:
CREATE OR REPLACE TEMPORARY VIEW global_runtime_dates AS
SELECT 
    /* ELAPRASE */
    DATE('2020-08-01') AS elaprase_dx_start_date,
    DATE('2023-08-01') AS elaprase_tx_start_date,

    /* AVLAYAH */
    -- DATE('2022-08-01') AS avlayah_dx_start_date,
    DATE('2026-03-01') AS avlayah_tx_start_date;

In [0]:
CREATE OR REPLACE TEMP VIEW runtime_parameters AS

SELECT
    (SELECT MAX(service_date) FROM com_edp_prd.com_raw.kom_medical_events) AS max_medical_date,

    (SELECT MAX(fill_date) FROM com_edp_prd.com_raw.kom_pharmacy_events) AS max_pharmacy_date,

    LAST_DAY(
        ADD_MONTHS(
            LEAST(
                (SELECT MAX(service_date) FROM com_edp_prd.com_raw.kom_medical_events),
                (SELECT MAX(fill_date) FROM com_edp_prd.com_raw.kom_pharmacy_events)
            ), -1
        )
    ) AS end_date,

    CURRENT_DATE() AS run_date;

    SELECT * FROM runtime_parameters;

In [0]:
CREATE OR REPLACE TEMP VIEW patient_hcp_visit_summary_Elaprase AS
WITH

MPSII_Diagnoses_Specified AS (
    SELECT DISTINCT PATIENT_ID, SERVICE_DATE AS FILL_DATE
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE DIAGNOSIS_CODES LIKE '%E761%'
      AND SERVICE_DATE BETWEEN (select elaprase_dx_start_date FROM global_runtime_dates) AND (SELECT end_date FROM runtime_parameters)
    UNION
    SELECT DISTINCT PATIENT_ID, FILL_DATE
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE DIAGNOSIS_CODE = 'E761'
      AND TRANSACTION_STATUS = 'PAID'
      AND FILL_DATE BETWEEN (select elaprase_dx_start_date FROM global_runtime_dates) AND (SELECT end_date FROM runtime_parameters)
),

-- Keep patients with >=2 distinct Dx dates for "Specified".
Patients_2Dx_Specified AS (
    SELECT PATIENT_ID
    FROM MPSII_Diagnoses_Specified
    GROUP BY PATIENT_ID
    HAVING COUNT(DISTINCT FILL_DATE) >= 2
),

-- Pull all "Unspecified" diagnosis events (E763) within the same window for Dx counting.
MPSII_Diagnoses_Unspecified AS (
    SELECT DISTINCT PATIENT_ID, SERVICE_DATE AS FILL_DATE
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE DIAGNOSIS_CODES LIKE '%E763%'
      AND SERVICE_DATE BETWEEN (select elaprase_dx_start_date FROM global_runtime_dates) AND (SELECT end_date FROM runtime_parameters)
    UNION
    SELECT DISTINCT PATIENT_ID, FILL_DATE
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE DIAGNOSIS_CODE = 'E763'
      AND TRANSACTION_STATUS = 'PAID'
      AND FILL_DATE BETWEEN (select elaprase_dx_start_date FROM global_runtime_dates) AND (SELECT end_date FROM runtime_parameters)
),

-- Keep patients with >=2 distinct Dx dates for "Unspecified".
Patients_2Dx_Unspecified AS (
    SELECT PATIENT_ID
    FROM MPSII_Diagnoses_Unspecified
    GROUP BY PATIENT_ID
    HAVING COUNT(DISTINCT FILL_DATE) >= 2
),

-- Treatment evidence universe (broad): Elaprase NDCs OR relevant infusion/procedure codes.

MPSII_Treatment_All AS (
    SELECT DISTINCT PATIENT_ID
    FROM (
        SELECT DISTINCT PATIENT_ID
        FROM com_edp_prd.com_raw.kom_medical_events
        WHERE NDC11 IN ('54092070001','540920700')
          AND SERVICE_DATE BETWEEN (select elaprase_tx_start_date FROM global_runtime_dates) AND (SELECT end_date FROM runtime_parameters)

        UNION ALL

        SELECT DISTINCT PATIENT_ID
        FROM com_edp_prd.com_raw.kom_pharmacy_events
        WHERE NDC11 IN ('54092070001','540920700')
          AND TRANSACTION_RESULT = 'PAID'
          AND FILL_DATE BETWEEN (select elaprase_tx_start_date FROM global_runtime_dates) AND (SELECT end_date FROM runtime_parameters)

        UNION ALL

        SELECT DISTINCT PATIENT_ID
        FROM com_edp_prd.com_raw.kom_medical_events
        -- WHERE PROCEDURE_CODE IN (
        --     '99601','99602','96365','96366','J1743','S9357','S9379',
        --     '38206','38230','38232','38240','38241','38242','38243','38250'
        -- )
        WHERE PROCEDURE_CODE IN ('J1743')
          AND SERVICE_DATE BETWEEN (select elaprase_tx_start_date FROM global_runtime_dates) AND (SELECT end_date FROM runtime_parameters)

        -- UNION ALL

        -- SELECT DISTINCT PATIENT_ID
        -- FROM com_edp_prd.com_raw.kom_medical_events
        -- WHERE PROCEDURE_CODE IN ('J3490', 'J3590', 'J9999')
        --   AND SERVICE_DATE >= (select avlayah_tx_start_date FROM global_runtime_dates)
    ) t
),

-- Treatment evidence (narrow): Elaprase only (NDCs + J1743).
-- Used for incremental inclusion of "Unspecified" cohort.
MPSII_Treatment_Elaprase_Only AS (
    SELECT DISTINCT PATIENT_ID
    FROM (
        SELECT DISTINCT PATIENT_ID
        FROM com_edp_prd.com_raw.kom_medical_events
        WHERE NDC11 IN ('54092070001','540920700')
          AND SERVICE_DATE BETWEEN (select elaprase_tx_start_date FROM global_runtime_dates) AND (SELECT end_date FROM runtime_parameters)

        UNION ALL

        SELECT DISTINCT PATIENT_ID
        FROM com_edp_prd.com_raw.kom_pharmacy_events
        WHERE NDC11 IN ('54092070001','540920700')
          AND TRANSACTION_RESULT = 'PAID'
          AND FILL_DATE BETWEEN (select elaprase_tx_start_date FROM global_runtime_dates) AND (SELECT end_date FROM runtime_parameters)

        UNION ALL

        SELECT DISTINCT PATIENT_ID
        FROM com_edp_prd.com_raw.kom_medical_events
        WHERE PROCEDURE_CODE = 'J1743'
          AND SERVICE_DATE BETWEEN (select elaprase_tx_start_date FROM global_runtime_dates) AND (SELECT end_date FROM runtime_parameters)

        -- UNION ALL

        -- SELECT DISTINCT PATIENT_ID
        -- FROM com_edp_prd.com_raw.kom_medical_events
        -- WHERE PROCEDURE_CODE IN ('J3490', 'J3590', 'J9999')
        --   AND SERVICE_DATE >= (select avlayah_tx_start_date FROM global_runtime_dates)
    ) t
),

-- Eligible "Specified" = >=2 Dx dates AND any treatment evidence.
Patients_2Dx_Specified_With_Treatment AS (
    SELECT DISTINCT p.PATIENT_ID
    FROM Patients_2Dx_Specified p
    INNER JOIN MPSII_Treatment_All t USING (PATIENT_ID)
),

-- Eligible "Incremental Unspecified" = >=2 Dx dates AND Elaprase-only evidence,
-- excluding anyone already in the specified+treatment set.
Patients_Incremental_Unspecified AS (
    SELECT DISTINCT p.PATIENT_ID
    FROM Patients_2Dx_Unspecified p
    INNER JOIN MPSII_Treatment_Elaprase_Only t USING (PATIENT_ID)
    WHERE p.PATIENT_ID NOT IN (SELECT PATIENT_ID FROM Patients_2Dx_Specified_With_Treatment)
),

-- Final eligible patient list.
eligible_patients AS (
    SELECT PATIENT_ID FROM Patients_2Dx_Specified_With_Treatment
    UNION
    SELECT PATIENT_ID FROM Patients_Incremental_Unspecified
)
select * from eligible_patients

In [0]:
select distinct Patient_ID from patient_hcp_visit_summary_Elaprase

In [0]:
WITH patient_list AS (
    SELECT col1 AS patient_id
    FROM VALUES
    ('QLPDH514'), ('7W4E0S3C'), ('GN3S2MG1'), ('GYR5QB5F'), ('GW0SVPYB'),
    ('D23P3T99'), ('4Q9YWTW2'), ('HWNPCZTZ'), ('WFGQ52KS'), ('8EHZX3LJ'),
    ('YVGNY6R8'), ('QZY771BV'), ('07F040KL'), ('N64CKZ1Y'), ('XSV8CB8R'),
    ('7T832LXS'), ('RSD92JLE'), ('N0H2F4RM'), ('4QQ3B4V7'), ('6F33PXJK'),
    ('DB60ZF3T'), ('9GQLZ50J'), ('8TQK891K'), ('BS6Z00T7'), ('YCV4YMKR'),
    ('V8DRGT1B'), ('GESKBZTW'), ('HG5089NV'), ('G5ND5VXN'), ('R99B343J'),
    ('P1JF1QND'), ('D9J5NQD4'), ('9T9D5NWE'), ('W7YZ8CSE'), ('NFC557NB'),
    ('ZF8XNQ6L'), ('EG39S37N'), ('FXS93NQW'), ('Q5HRM4MF'), ('P9LPDJJ2'),
    ('ZPP65974'), ('VM0F6LD8'), ('PHDFWCCN'), ('CPV0FW0X'), ('HRR7CK0R'),
    ('RX61Q59G'), ('ZLXV6L4G'), ('41J6Q2LG'), ('VL3L6GFC'), ('GCGPRX9R'),
    ('RN7HW77Q'), ('36LNY5TR'), ('186FMXEG'), ('X4HTR4LM'), ('WYB4NG8H'),
    ('H6S5SSXY'), ('35WZEPN0'), ('Q3TSJ19Q'), ('DR65XZXF'), ('BTEQP2RC'),
    ('HFDG1Z0F'), ('ER85PYPC'), ('XT8CBPT4'), ('BKVRTPS7'), ('VQV4TSJQ'),
    ('1H6M3WFH'), ('M9XZMW52'), ('JJ5H8T2H'), ('R03CVQFW'), ('BDV32518'),
    ('52028LXY'), ('HPBHMXNM'), ('0FD2MVJH'), ('YPHZSNH8'), ('9YVY62P8'),
    ('6K3MNJ8Z'), ('6XZ63TZJ'), ('PC0DTBM6'), ('4NNDX85F'), ('6513VL4S'),
    ('7BDN8F8G'), ('GZXJ6JRM'), ('YE49YKQX'), ('WQMGYPEG'), ('HNNWFT61'),
    ('C4303LEZ'), ('C6S52DKD'), ('8QR2GY9W'), ('VCFSGYCB'), ('4969V5BL'),
    ('E79M532Q'), ('WCLK5QGP'), ('R0BT2KNH'), ('9J08HQPH'), ('55MXFFR4'),
    ('ERZKNJ8N'), ('8WW54LY8'), ('CLXBXFXL'), ('ND2Y0LRD'), ('T84CH62W'),
    ('D7KL4DYC'), ('5BWJRK84'), ('YTP52KP4'), ('LJFTZB0F'), ('R6BP8FQ1'),
    ('DKVQZJQB'), ('QVN33L82'), ('2W1LYFK2'), ('6MGD17QW'), ('35RJEZJM'),
    ('T1C668E6'), ('FECLC9C3'), ('REQPDYJ6'), ('Y9GW7HPB'), ('DNV974CH'),
    ('HTZV1K7F')
),

elaprase_flag AS (
    SELECT DISTINCT patient_id
    FROM (
        SELECT PATIENT_ID
        FROM com_edp_prd.com_raw.kom_medical_events
        WHERE NDC11 IN ('54092070001','540920700')
           OR PROCEDURE_CODE = 'J1743'

        UNION

        SELECT PATIENT_ID
        FROM com_edp_prd.com_raw.kom_pharmacy_events
        WHERE NDC11 IN ('54092070001','540920700')
          AND TRANSACTION_RESULT = 'PAID'
    )
)

SELECT 
    pl.patient_id,
    CASE 
        WHEN ef.patient_id IS NOT NULL THEN 1 
        ELSE 0 
    END AS ever_elaprase_flag,
    p360.*
FROM patient_list pl
LEFT JOIN com_edp_prd.cmpa_insights_internal_schema.patient360_master p360
    ON pl.patient_id = p360.patient_id
LEFT JOIN elaprase_flag ef
    ON pl.patient_id = ef.patient_id;

In [0]:
CREATE OR REPLACE TEMP VIEW patient_hcp_visit_summary_Elaprase_New AS

WITH MPSII_Diagnoses_Specified AS (

    SELECT DISTINCT
        PATIENT_ID,
        SERVICE_DATE AS DX_DATE
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE DIAGNOSIS_CODES LIKE '%E761%'

    UNION

    SELECT DISTINCT
        PATIENT_ID,
        FILL_DATE AS DX_DATE
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE DIAGNOSIS_CODE = 'E761'
      AND TRANSACTION_STATUS = 'PAID'
),

Patients_2Dx_Specified AS (

    SELECT PATIENT_ID
    FROM MPSII_Diagnoses_Specified
    GROUP BY PATIENT_ID
    HAVING COUNT(DISTINCT DX_DATE) >= 2
),

MPSII_Treatment_Elaprase_Ever AS (

    SELECT DISTINCT PATIENT_ID
    FROM (

        SELECT DISTINCT PATIENT_ID
        FROM com_edp_prd.com_raw.kom_medical_events
        WHERE NDC11 IN ('54092070001','540920700')

        UNION ALL

        SELECT DISTINCT PATIENT_ID
        FROM com_edp_prd.com_raw.kom_pharmacy_events
        WHERE NDC11 IN ('54092070001','540920700')
          AND TRANSACTION_RESULT = 'PAID'

        UNION ALL

        SELECT DISTINCT PATIENT_ID
        FROM com_edp_prd.com_raw.kom_medical_events
        WHERE PROCEDURE_CODE = 'J1743'

    ) t
),

Base_MPSII_Universe AS (

    SELECT PATIENT_ID
    FROM Patients_2Dx_Specified

    UNION 

    SELECT PATIENT_ID
    FROM MPSII_Treatment_Elaprase_Ever
),

Recent_Treatment_2Y AS (

    SELECT DISTINCT PATIENT_ID
    FROM (

        -- Elaprase NDC
        SELECT DISTINCT PATIENT_ID
        FROM com_edp_prd.com_raw.kom_medical_events
        WHERE NDC11 IN ('54092070001','540920700')
          AND SERVICE_DATE >= (SELECT elaprase_tx_start_date FROM global_runtime_dates)

        UNION ALL

        SELECT DISTINCT PATIENT_ID
        FROM com_edp_prd.com_raw.kom_pharmacy_events
        WHERE NDC11 IN ('54092070001','540920700')
          AND TRANSACTION_RESULT = 'PAID'
          AND FILL_DATE >= (SELECT elaprase_tx_start_date FROM global_runtime_dates)

        UNION ALL

        -- Elaprase J-code
        SELECT DISTINCT PATIENT_ID
        FROM com_edp_prd.com_raw.kom_medical_events
        WHERE PROCEDURE_CODE = 'J1743'
          AND SERVICE_DATE >= (SELECT elaprase_tx_start_date FROM global_runtime_dates)
        
        UNION ALL

        -- Other ERT Procedure Codes
        SELECT DISTINCT PATIENT_ID
        FROM com_edp_prd.com_raw.kom_medical_events
        WHERE PROCEDURE_CODE IN ('99601','99602','96365','96366','S9357','S9379','38206','38230','38232','38240','38241','38242','38243','38250')
          AND SERVICE_DATE >= (SELECT elaprase_tx_start_date FROM global_runtime_dates)

    ) t
),

eligible_patients AS (

    SELECT DISTINCT b.PATIENT_ID
    FROM Base_MPSII_Universe b
    INNER JOIN Recent_Treatment_2Y r
        ON b.PATIENT_ID = r.PATIENT_ID
)

SELECT *
FROM eligible_patients;

In [0]:
-- =============================================================================
-- STEP 1: Eligible Patient Cohort WITH FLAGS
-- =============================================================================

CREATE OR REPLACE TEMP VIEW eligible_patients_new AS

WITH MPSII_Diagnoses_Specified AS (

    SELECT DISTINCT
        PATIENT_ID,
        SERVICE_DATE AS DX_DATE
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE DIAGNOSIS_CODES LIKE '%E761%'

    UNION

    SELECT DISTINCT
        PATIENT_ID,
        FILL_DATE AS DX_DATE
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE DIAGNOSIS_CODE = 'E761'
      AND TRANSACTION_RESULT = 'PAID'
),

Patients_2Dx_Specified AS (

    SELECT PATIENT_ID
    FROM MPSII_Diagnoses_Specified
    GROUP BY PATIENT_ID
    HAVING COUNT(DISTINCT DX_DATE) >= 2
),

MPSII_Treatment_Elaprase_Ever AS (

    SELECT DISTINCT PATIENT_ID
    FROM (

        -- Medical NDC
        SELECT DISTINCT PATIENT_ID
        FROM com_edp_prd.com_raw.kom_medical_events
        WHERE NDC11 IN ('54092070001','540920700')

        UNION ALL

        -- Pharmacy NDC
        SELECT DISTINCT PATIENT_ID
        FROM com_edp_prd.com_raw.kom_pharmacy_events
        WHERE NDC11 IN ('54092070001','540920700')
          AND TRANSACTION_RESULT = 'PAID'

        UNION ALL

        -- J-code
        SELECT DISTINCT PATIENT_ID
        FROM com_edp_prd.com_raw.kom_medical_events
        WHERE PROCEDURE_CODE = 'J1743'

    ) t
),

Base_MPSII_Universe AS (

    SELECT
        PATIENT_ID,
        1 AS FLAG_2DX,
        0 AS FLAG_TX_EVER
    FROM Patients_2Dx_Specified

    UNION

    SELECT
        PATIENT_ID,
        0 AS FLAG_2DX,
        1 AS FLAG_TX_EVER
    FROM MPSII_Treatment_Elaprase_Ever
),

Base_MPSII_Universe_Final AS (

    SELECT
        PATIENT_ID,
        MAX(FLAG_2DX) AS FLAG_2DX,
        MAX(FLAG_TX_EVER) AS FLAG_TX_EVER
    FROM Base_MPSII_Universe
    GROUP BY PATIENT_ID
),

Recent_Treatment_2Y AS (

    SELECT DISTINCT PATIENT_ID
    FROM (

        -- Elaprase NDC (medical)
        SELECT DISTINCT PATIENT_ID
        FROM com_edp_prd.com_raw.kom_medical_events
        WHERE NDC11 IN ('54092070001','540920700')
          AND SERVICE_DATE >= (
                SELECT elaprase_tx_start_date
                FROM global_runtime_dates
          )

        UNION ALL

        -- Elaprase NDC (pharmacy)
        SELECT DISTINCT PATIENT_ID
        FROM com_edp_prd.com_raw.kom_pharmacy_events
        WHERE NDC11 IN ('54092070001','540920700')
          AND TRANSACTION_RESULT = 'PAID'
          AND FILL_DATE >= (
                SELECT elaprase_tx_start_date
                FROM global_runtime_dates
          )

        UNION ALL

        -- Elaprase J-code
        SELECT DISTINCT PATIENT_ID
        FROM com_edp_prd.com_raw.kom_medical_events
        WHERE PROCEDURE_CODE = 'J1743'
          AND SERVICE_DATE >= (
                SELECT elaprase_tx_start_date
                FROM global_runtime_dates
          )

        UNION ALL

        -- Other infusion / transplant procedure codes
        SELECT DISTINCT PATIENT_ID
        FROM com_edp_prd.com_raw.kom_medical_events
        WHERE PROCEDURE_CODE IN (
            '99601','99602','96365','96366',
            'S9357','S9379',
            '38206','38230','38232',
            '38240','38241','38242',
            '38243','38250'
        )
          AND SERVICE_DATE >= (
                SELECT elaprase_tx_start_date
                FROM global_runtime_dates
          )

    ) t
),

eligible_patients AS (

    SELECT DISTINCT
        b.PATIENT_ID,

        b.FLAG_2DX,
        b.FLAG_TX_EVER,

    CASE
        WHEN b.FLAG_2DX = 1
            AND r.PATIENT_ID IS NOT NULL
        THEN 1
        ELSE 0
    END AS FLAG_2DX_AND_RECENT_TX,

    CASE
        WHEN b.FLAG_TX_EVER = 1
            AND r.PATIENT_ID IS NOT NULL
        THEN 1
        ELSE 0
    END AS FLAG_TXEVER_AND_RECENT_TX

    FROM Base_MPSII_Universe_Final b

    INNER JOIN Recent_Treatment_2Y r
        ON b.PATIENT_ID = r.PATIENT_ID
)

SELECT *
FROM eligible_patients;

-- =============================================================================
-- STEP 2: Provider filter
-- =============================================================================

CREATE OR REPLACE TEMP VIEW cohort_3_learnings_new AS

SELECT DISTINCT npi
FROM com_raw.kom_providers
WHERE provider_type = 'INDIVIDUAL'
  AND (
      PRIMARY_SPECIALTY NOT IN (
        'Anesthesiologist Assistant',
        'Anesthesiology',
        'Dentist',
        'Dietitian, Registered',
        'Emergency Medical Technician, Basic',
        'Emergency Medicine',
        'General Acute Care Hospital',
        'Nurse Anesthetist, Certified Registered',
        'Obstetrics & Gynecology',
        'Pathology',
        'Radiology',
        'Urology'
      )

      OR SECONDARY_SPECIALTY IN (
        'Child & Adolescent Psychiatry',
        'Psychiatry',
        'Adolescent Medicine',
        'Developmental - Behavioral Pediatrics',
        'Neonatal-Perinatal Medicine',
        'Nutrition, Pediatric',
        'Oncology, Pediatrics',
        'Pediatric Cardiology',
        'Pediatric Critical Care Medicine',
        'Pediatric Dermatology',
        'Pediatric Emergency Medicine',
        'Pediatric Endocrinology',
        'Pediatric Gastroenterology',
        'Pediatric Hematology-Oncology',
        'Pediatric Infectious Diseases',
        'Pediatric Nephrology',
        'Pediatric Ophthalmology and Strabismus Specialist',
        'Pediatric Orthopaedic Surgery',
        'Pediatric Otolaryngology',
        'Pediatric Pulmonology',
        'Pediatric Radiology',
        'Pediatric Rehabilitation Medicine',
        'Pediatric Rheumatology',
        'Pediatric Surgery',
        'Pediatrics',
        'Clinical Biochemical Genetics',
        'Clinical Genetics (M.D.)',
        'Clinical Molecular Genetics',
        'Ph.D. Medical Genetics',
        'Neurodevelopmental Disabilities',
        'Neurology',
        'Neurology with Special Qualifications in Child Neurology',
        'Neuroradiology'
      )
  );


-- =============================================================================
-- STEP 3: Build all_patient_claims_new directly
-- (No separate Step 3/4/5 temp views)
-- =============================================================================

CREATE OR REPLACE TEMP VIEW all_patient_claims_new AS

SELECT *
FROM (

    -- =========================
    -- DX CLAIMS - MEDICAL
    -- =========================
    SELECT DISTINCT
        PATIENT_ID,
        COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI,
        SERVICE_DATE AS FILL_DATE,
        'DX' AS CLAIM_TYPE
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE (
            DIAGNOSIS_CODES LIKE '%E761%'
          )
      AND PATIENT_ID IN (
            SELECT PATIENT_ID
            FROM eligible_patients_new
      )

    UNION

    -- =========================
    -- DX CLAIMS - PHARMACY
    -- =========================
    SELECT DISTINCT
        PATIENT_ID,
        PRESCRIBER_NPI AS NPI,
        FILL_DATE,
        'DX' AS CLAIM_TYPE
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE DIAGNOSIS_CODE IN ('E761')
      AND TRANSACTION_RESULT = 'PAID'
      AND PATIENT_ID IN (
            SELECT PATIENT_ID
            FROM eligible_patients_new
      )

    UNION

    -- =========================
    -- TX CLAIMS - MEDICAL NDC
    -- =========================
    SELECT DISTINCT
        PATIENT_ID,
        COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI,
        SERVICE_DATE AS FILL_DATE,
        'TX' AS CLAIM_TYPE
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE NDC11 IN ('54092070001','540920700')
      AND PATIENT_ID IN (
            SELECT PATIENT_ID
            FROM eligible_patients_new
      )

    UNION

    -- =========================
    -- TX CLAIMS - PROCEDURE
    -- =========================
    SELECT DISTINCT
        PATIENT_ID,
        RENDERING_NPI AS NPI,
        SERVICE_DATE AS FILL_DATE,
        'TX' AS CLAIM_TYPE
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE PROCEDURE_CODE IN (
        '99601','99602','96365','96366',
        'J1743','S9357','S9379',
        '38206','38230','38232',
        '38240','38241','38242',
        '38243','38250'
    )
      AND PATIENT_ID IN (
            SELECT PATIENT_ID
            FROM eligible_patients_new
      )

    UNION

    -- =========================
    -- TX CLAIMS - PHARMACY
    -- =========================
    SELECT DISTINCT
        PATIENT_ID,
        PRESCRIBER_NPI AS NPI,
        FILL_DATE,
        'TX' AS CLAIM_TYPE
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE NDC11 IN ('54092070001','540920700')
      AND TRANSACTION_RESULT = 'PAID'
      AND PATIENT_ID IN (
            SELECT PATIENT_ID
            FROM eligible_patients_new
      )

) a

WHERE NPI IN (
    SELECT DISTINCT NPI
    FROM cohort_3_learnings_new
);


-- =============================================================================
-- STEP 4: Primary HCP Assignment
-- =============================================================================

CREATE OR REPLACE TEMP VIEW primary_hcp_new AS

WITH hcp_metrics AS (

    SELECT
        a.PATIENT_ID,
        a.NPI,

        CASE
            WHEN p.primary_specialty LIKE '%Genetic%'
              OR p.secondary_specialty LIKE '%Genetic%'
                THEN 'Geneticist'

            WHEN p.primary_specialty LIKE '%Psychiatry & Neurology%'
              OR p.secondary_specialty LIKE '%Neurodevelopmental Disabilities%'
              OR p.primary_specialty LIKE '%Neurological Surgery%'
                THEN 'Psychiatry & Neurology'

            WHEN p.primary_specialty LIKE '%Pediatrics%'
                THEN 'Pediatrician'

            WHEN p.primary_specialty LIKE '%Internal Medicine%'
              OR p.secondary_specialty LIKE '%Internal Medicine%'
              OR p.primary_specialty LIKE '%Family Medicine%'
              OR p.secondary_specialty LIKE '%Family Medicine%'
                THEN 'PCP'

            WHEN p.primary_specialty LIKE '%Nurse Practitioner%'
              OR p.primary_specialty LIKE '%Physician Assistant%'
                THEN 'NPPA'

            WHEN a.NPI IS NULL
                THEN 'NA'

            ELSE 'Others'
        END AS SPECIALTY,

        CASE
            WHEN p.primary_specialty LIKE '%Genetic%'
              OR p.secondary_specialty LIKE '%Genetic%'
                THEN 1

            WHEN p.primary_specialty LIKE '%Psychiatry & Neurology%'
              OR p.secondary_specialty LIKE '%Neurodevelopmental Disabilities%'
              OR p.primary_specialty LIKE '%Neurological Surgery%'
                THEN 2

            WHEN p.primary_specialty LIKE '%Pediatrics%'
                THEN 3

            WHEN p.primary_specialty LIKE '%Internal Medicine%'
              OR p.secondary_specialty LIKE '%Internal Medicine%'
              OR p.primary_specialty LIKE '%Family Medicine%'
              OR p.secondary_specialty LIKE '%Family Medicine%'
                THEN 4

            WHEN p.primary_specialty LIKE '%Nurse Practitioner%'
              OR p.primary_specialty LIKE '%Physician Assistant%'
                THEN 5

            WHEN a.NPI IS NULL
                THEN 7

            ELSE 6
        END AS SPECIALTY_PRIORITY,

        COUNT(DISTINCT a.FILL_DATE) AS NO_OF_VISITS,

        COUNT(DISTINCT CASE WHEN a.CLAIM_TYPE = 'DX' THEN a.FILL_DATE END) AS DX_VISITS,

        COUNT(DISTINCT CASE WHEN a.CLAIM_TYPE = 'TX' THEN a.FILL_DATE END) AS TX_VISITS,

        MAX(a.FILL_DATE) AS MOST_RECENT_VISIT

    FROM all_patient_claims_new a

    LEFT JOIN com_edp_prd.com_raw.kom_providers p
        ON a.NPI = p.NPI

    GROUP BY
        a.PATIENT_ID,
        a.NPI,
        p.primary_specialty,
        p.secondary_specialty
),

ranked_hcps AS (

    SELECT
        *,

        RANK() OVER (
            PARTITION BY PATIENT_ID
            ORDER BY
                SPECIALTY_PRIORITY ASC,
                NO_OF_VISITS DESC,
                MOST_RECENT_VISIT DESC,
                NPI ASC
        ) AS HCP_RANK

    FROM hcp_metrics
)

SELECT
    PATIENT_ID,
    NPI AS PRIMARY_HCP_NPI,
    SPECIALTY AS PRIMARY_HCP_SPECIALTY,
    SPECIALTY_PRIORITY,
    NO_OF_VISITS,
    DX_VISITS,
    TX_VISITS,
    MOST_RECENT_VISIT,
    HCP_RANK
FROM ranked_hcps
WHERE HCP_RANK = 1;


-- =============================================================================
-- STEP 5: Materialize enriched primary_hcp_new table
-- =============================================================================

CREATE OR REPLACE TABLE com_edp_prd.cmpa_insights_internal_schema.primary_hcp_new AS

SELECT
    ph.*,

    COALESCE(p.FIRST_NAME, '') || ' ' || COALESCE(p.LAST_NAME, '')
        AS primary_hcp_name_2yr,

    ref.HCO_NAME AS primary_hcp_hco_name_2yr,
    ref.HCO_CITY AS primary_hcp_hco_city_2yr,
    ref.HCO_STATE AS primary_hcp_hco_state_2yr,

    ref.mapped_territory_id AS primary_hcp_territory_id_2yr,
    ref.TERRITORY           AS primary_hcp_territory_2yr,

    ref.mapped_region_id    AS primary_hcp_region_id_2yr,
    ref.region              AS primary_hcp_region_2yr

FROM primary_hcp_new ph

LEFT JOIN com_edp_prd.com_raw.kom_providers p
    ON ph.PRIMARY_HCP_NPI = p.NPI

LEFT JOIN (

    SELECT
        a.hcp_npi,
        a.hco_name,
        a.hco_city,
        a.hco_state,
        a.territory,
        a.region,

        b.territory_id AS mapped_territory_id,
        c.region_id    AS mapped_region_id,

        a.hcp_primary_specialty AS hcp_specialty

    FROM cmpa_insights_internal_schema.reference_file a

    LEFT JOIN (
        SELECT DISTINCT
            TRY_CAST(territory_id AS BIGINT) AS territory_id,
            territory_name
        FROM cmpa_insights_internal_schema.zip_to_territory_mapping
    ) b
        ON TRY_CAST(a.territory_id AS BIGINT) = b.territory_id

    LEFT JOIN (
        SELECT DISTINCT
            TRY_CAST(region_id AS BIGINT) AS region_id,
            region_name
        FROM cmpa_insights_internal_schema.zip_to_territory_mapping
    ) c
        ON TRY_CAST(a.region_id AS BIGINT) = c.region_id

) ref
    ON ph.PRIMARY_HCP_NPI = ref.hcp_npi;

CREATE OR REPLACE TABLE
com_edp_prd.cmpa_insights_internal_schema.patient360_master_new AS

WITH

-- =========================================
-- OLD PATIENT360 COHORT FLAG
-- =========================================
old_cohort_flag AS (

    SELECT DISTINCT
        PATIENT_ID,
        1 AS OLD_COHORT_FLAG
    FROM com_edp_prd.cmpa_insights_internal_schema.patient360_base
),

-- =========================================
-- DEMOGRAPHICS
-- =========================================
patient_demographics AS (

    SELECT *
    FROM (

        SELECT
            PATIENT_ID,
            PATIENT_YOB,
            YEAR(CURRENT_DATE()) - YEAR(PATIENT_YOB) AS PATIENT_AGE,
            PATIENT_GENDER,

            ROW_NUMBER() OVER (
                PARTITION BY PATIENT_ID
                ORDER BY PATIENT_YOB
            ) rn

        FROM com_edp_prd.com_raw.kom_patient_demographics
    )
    WHERE rn = 1
),

-- =========================================
-- GEOGRAPHY
-- =========================================
patient_geography AS (

    SELECT *
    FROM (

        SELECT
            PATIENT_ID,
            PATIENT_STATE,

            ROW_NUMBER() OVER (
                PARTITION BY PATIENT_ID
                ORDER BY VALID_TO_DATE DESC
            ) rn

        FROM com_edp_prd.com_raw.kom_patient_geography
    )
    WHERE rn = 1
),

-- =========================================
-- INCIDENCE DATE (DX ONLY)
-- =========================================
incidence_dates AS (

    SELECT
        PATIENT_ID,
        MIN(FILL_DATE) AS incidence_date

    FROM all_patient_claims_new
    WHERE CLAIM_TYPE = 'DX'

    GROUP BY PATIENT_ID
),

-- =========================================
-- TRUE ELAPRASE TX CLAIMS
-- =========================================
true_elaprase_tx_claims AS (

    SELECT
        PATIENT_ID,
        SERVICE_DATE AS TX_DATE,
        'ELAPRASE' AS TX_TYPE
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE NDC11 IN ('54092070001','540920700')

    UNION ALL

    SELECT
        PATIENT_ID,
        FILL_DATE AS TX_DATE,
        'ELAPRASE' AS TX_TYPE
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE NDC11 IN ('54092070001','540920700')
      AND TRANSACTION_RESULT = 'PAID'

    UNION ALL

    SELECT
        PATIENT_ID,
        SERVICE_DATE AS TX_DATE,
        'ELAPRASE' AS TX_TYPE
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE PROCEDURE_CODE = 'J1743'
),

-- =========================================
-- RECENT 2YR QUALIFYING TREATMENT CLAIMS
-- =========================================
recent_2yr_treatment_claims AS (

    -- Elaprase Medical NDC
    SELECT
        PATIENT_ID,
        SERVICE_DATE AS TX_DATE,
        'ELAPRASE' AS TX_TYPE
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE NDC11 IN ('54092070001','540920700')
      AND SERVICE_DATE >= (
            SELECT elaprase_tx_start_date
            FROM global_runtime_dates
      )

    UNION ALL

    -- Elaprase Pharmacy NDC
    SELECT
        PATIENT_ID,
        FILL_DATE AS TX_DATE,
        'ELAPRASE' AS TX_TYPE
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE NDC11 IN ('54092070001','540920700')
      AND TRANSACTION_RESULT = 'PAID'
      AND FILL_DATE >= (
            SELECT elaprase_tx_start_date
            FROM global_runtime_dates
      )

    UNION ALL

    -- Elaprase J-code
    SELECT
        PATIENT_ID,
        SERVICE_DATE AS TX_DATE,
        'ELAPRASE' AS TX_TYPE
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE PROCEDURE_CODE = 'J1743'
      AND SERVICE_DATE >= (
            SELECT elaprase_tx_start_date
            FROM global_runtime_dates
      )

    UNION ALL

    -- Other infusion / transplant procedure codes
    SELECT
        PATIENT_ID,
        SERVICE_DATE AS TX_DATE,
        'OTHER_TX' AS TX_TYPE
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE PROCEDURE_CODE IN (
        '99601','99602','96365','96366',
        'S9357','S9379',
        '38206','38230','38232',
        '38240','38241','38242',
        '38243','38250'
    )
      AND SERVICE_DATE >= (
            SELECT elaprase_tx_start_date
            FROM global_runtime_dates
      )
),

-- =========================================
-- ELAPRASE TX IN LAST 2 YEARS FLAG
-- =========================================
elaprase_last_2yr_flag AS (

    SELECT DISTINCT
        PATIENT_ID,
        1 AS FLAG_ELAPRASE_LAST_2YR

    FROM (

        -- Medical NDC
        SELECT DISTINCT
            PATIENT_ID
        FROM com_edp_prd.com_raw.kom_medical_events
        WHERE NDC11 IN ('54092070001','540920700')
          AND SERVICE_DATE >= (
                SELECT elaprase_tx_start_date
                FROM global_runtime_dates
          )

        UNION

        -- Pharmacy NDC
        SELECT DISTINCT
            PATIENT_ID
        FROM com_edp_prd.com_raw.kom_pharmacy_events
        WHERE NDC11 IN ('54092070001','540920700')
          AND TRANSACTION_RESULT = 'PAID'
          AND FILL_DATE >= (
                SELECT elaprase_tx_start_date
                FROM global_runtime_dates
          )

        UNION

        -- J-code
        SELECT DISTINCT
            PATIENT_ID
        FROM com_edp_prd.com_raw.kom_medical_events
        WHERE PROCEDURE_CODE = 'J1743'
          AND SERVICE_DATE >= (
                SELECT elaprase_tx_start_date
                FROM global_runtime_dates
          )

    ) x
),

-- =========================================
-- FIRST QUALIFYING TX DATE
-- =========================================
first_tx_dates AS (

    SELECT
        PATIENT_ID,
        MIN(TX_DATE) AS first_incidence_treatment_date

    FROM recent_2yr_treatment_claims
    GROUP BY PATIENT_ID
),

-- =========================================
-- LATEST QUALIFYING TX DATE
-- =========================================
latest_tx_dates AS (

    SELECT
        PATIENT_ID,
        MAX(TX_DATE) AS latest_treatment_date

    FROM recent_2yr_treatment_claims
    GROUP BY PATIENT_ID
),

-- =========================================
-- LATEST TX TYPE
-- =========================================
latest_tx_type AS (

    SELECT *
    FROM (

        SELECT
            PATIENT_ID,
            TX_TYPE AS latest_mpsii_tx_type,
            TX_DATE,

            ROW_NUMBER() OVER (
                PARTITION BY PATIENT_ID
                ORDER BY TX_DATE DESC
            ) rn

        FROM recent_2yr_treatment_claims
    )
    WHERE rn = 1
),

-- =========================================
-- LATEST CLAIM DATE
-- =========================================
latest_claim_dates AS (

    SELECT
        PATIENT_ID,
        MAX(CLAIM_DATE) AS latest_claim_date

    FROM (

        SELECT
            PATIENT_ID,
            SERVICE_DATE AS CLAIM_DATE
        FROM com_edp_prd.com_raw.kom_medical_events
        WHERE PATIENT_ID IN (
            SELECT PATIENT_ID
            FROM eligible_patients_new
        )

        UNION ALL

        SELECT
            PATIENT_ID,
            FILL_DATE AS CLAIM_DATE
        FROM com_edp_prd.com_raw.kom_pharmacy_events
        WHERE PATIENT_ID IN (
            SELECT PATIENT_ID
            FROM eligible_patients_new
        )
          AND TRANSACTION_RESULT = 'PAID'

    ) x

    GROUP BY PATIENT_ID
),

-- =========================================
-- ELAPRASE FILLS
-- =========================================
elaprase_fills AS (

    SELECT
        PATIENT_ID,
        COUNT(DISTINCT TX_DATE) AS elaprase_fills

    FROM true_elaprase_tx_claims
    GROUP BY PATIENT_ID
),

-- =========================================
-- ALL HCP VISITS
-- =========================================
hcp_visit_metrics AS (

    SELECT
        PATIENT_ID,
        NPI,
        COUNT(DISTINCT FILL_DATE) AS all_visit_count_5yr,
        COUNT(DISTINCT CASE WHEN CLAIM_TYPE = 'TX' THEN FILL_DATE END) AS treatment_visit_count_5yr,
        MAX(FILL_DATE) AS last_visit_5yr

    FROM all_patient_claims_new
    GROUP BY PATIENT_ID, NPI
),

-- =========================================
-- FIRST DX HCP
-- =========================================
first_dx_hcp AS (

    SELECT *
    FROM (

        SELECT
            a.PATIENT_ID,
            a.NPI AS first_dx_hcp_5yr,
            h.all_visit_count_5yr AS first_dx_all_visit_count_5yr,
            h.last_visit_5yr AS first_dx_last_visit_5yr,

            ROW_NUMBER() OVER (
                PARTITION BY a.PATIENT_ID
                ORDER BY a.FILL_DATE ASC, a.NPI
            ) rn

        FROM all_patient_claims_new a

        LEFT JOIN hcp_visit_metrics h
            ON a.PATIENT_ID = h.PATIENT_ID
           AND a.NPI = h.NPI

        WHERE a.CLAIM_TYPE = 'DX'
    )
    WHERE rn = 1
),

-- =========================================
-- FIRST TX HCP
-- =========================================
first_tx_hcp AS (

    SELECT *
    FROM (

        SELECT
            a.PATIENT_ID,
            a.NPI AS first_tx_hcp_5yr,
            h.all_visit_count_5yr AS first_tx_all_visit_count_5yr,
            h.treatment_visit_count_5yr AS first_tx_treatment_visit_count_5yr,
            h.last_visit_5yr AS first_tx_last_visit_5yr,

            ROW_NUMBER() OVER (
                PARTITION BY a.PATIENT_ID
                ORDER BY a.FILL_DATE ASC, a.NPI
            ) rn

        FROM all_patient_claims_new a

        LEFT JOIN hcp_visit_metrics h
            ON a.PATIENT_ID = h.PATIENT_ID
           AND a.NPI = h.NPI

        WHERE a.CLAIM_TYPE = 'TX'
    )
    WHERE rn = 1
),

-- =========================================
-- MOST SEEN HCPS
-- =========================================
most_seen_hcps AS (

    SELECT
        PATIENT_ID,
        NPI,
        all_visit_count_5yr,
        last_visit_5yr,

        ROW_NUMBER() OVER (
            PARTITION BY PATIENT_ID
            ORDER BY all_visit_count_5yr DESC,
                     last_visit_5yr DESC,
                     NPI
        ) AS visit_rank

    FROM hcp_visit_metrics
),

-- =========================================
-- LATEST CLAIM HCP
-- =========================================
latest_claim_hcp AS (

    SELECT *
    FROM (

        SELECT
            a.PATIENT_ID,
            a.NPI AS latest_claim_hcp_npi,
            p.FIRST_NAME,
            p.LAST_NAME,
            p.PRIMARY_SPECIALTY,
            h.all_visit_count_5yr,
            ref.hco_name,

            ROW_NUMBER() OVER (
                PARTITION BY a.PATIENT_ID
                ORDER BY a.FILL_DATE DESC
            ) rn

        FROM all_patient_claims_new a

        LEFT JOIN com_edp_prd.com_raw.kom_providers p
            ON a.NPI = p.NPI

        LEFT JOIN hcp_visit_metrics h
            ON a.PATIENT_ID = h.PATIENT_ID
           AND a.NPI = h.NPI

        LEFT JOIN cmpa_insights_internal_schema.reference_file ref
            ON a.NPI = ref.hcp_npi
    )
    WHERE rn = 1
),

-- =========================================
-- LATEST TX HCP
-- =========================================
latest_treatment_hcp AS (

    SELECT *
    FROM (

        SELECT
            a.PATIENT_ID,
            a.NPI AS latest_treatment_hcp_npi,
            p.FIRST_NAME,
            p.LAST_NAME,
            p.PRIMARY_SPECIALTY,
            h.all_visit_count_5yr,
            ref.hco_name,

            ROW_NUMBER() OVER (
                PARTITION BY a.PATIENT_ID
                ORDER BY a.FILL_DATE DESC
            ) rn

        FROM all_patient_claims_new a

        LEFT JOIN com_edp_prd.com_raw.kom_providers p
            ON a.NPI = p.NPI

        LEFT JOIN hcp_visit_metrics h
            ON a.PATIENT_ID = h.PATIENT_ID
           AND a.NPI = h.NPI

        LEFT JOIN cmpa_insights_internal_schema.reference_file ref
            ON a.NPI = ref.hcp_npi

        WHERE a.CLAIM_TYPE = 'TX'
    )
    WHERE rn = 1
)

SELECT DISTINCT

    ep.PATIENT_ID,

    pd.PATIENT_YOB,
    pd.PATIENT_AGE,
    pd.PATIENT_GENDER,
    pg.PATIENT_STATE AS patient_state,

    idx.incidence_date,

    ftx.first_incidence_treatment_date,

    lcd.latest_claim_date,

    lch.latest_claim_hcp_npi,

    COALESCE(lch.FIRST_NAME,'') || ' ' || COALESCE(lch.LAST_NAME,'')
        AS latest_claim_hcp_name,

    lch.PRIMARY_SPECIALTY AS latest_claim_hcp_specialty,

    lch.all_visit_count_5yr AS latest_claim_hcp_visit_count,

    lch.hco_name AS latest_claim_hcp_hco_name,

    ltd.latest_treatment_date,

    ltt.latest_mpsii_tx_type,

    CASE
        WHEN ftx.first_incidence_treatment_date >= idx.incidence_date
        THEN 1
        ELSE 0
    END AS first_tx_after_diagnosis,

    CASE
        WHEN idx.incidence_date IS NOT NULL
         AND ftx.first_incidence_treatment_date IS NOT NULL
        THEN ROUND(
                MONTHS_BETWEEN(
                    ftx.first_incidence_treatment_date,
                    idx.incidence_date
                ),
                0
             )
    END AS time_dx_to_first_tx_in_months,

    CASE
        WHEN ltd.latest_treatment_date IS NOT NULL
             AND ftx.first_incidence_treatment_date IS NOT NULL
        THEN ROUND(
                MONTHS_BETWEEN(
                    ltd.latest_treatment_date,
                    ftx.first_incidence_treatment_date
                ),
                0
             )
    END AS treatment_period_months,

    COALESCE(ef.elaprase_fills,0) AS elaprase_fills,
    
    lth.latest_treatment_hcp_npi,

    COALESCE(lth.FIRST_NAME,'') || ' ' || COALESCE(lth.LAST_NAME,'')
        AS latest_treatment_hcp_name,

    lth.PRIMARY_SPECIALTY AS latest_treatment_hcp_specialty,

    lth.all_visit_count_5yr AS latest_treatment_hcp_visit_count,

    lth.hco_name AS latest_treatment_hcp_hco_name,

    fdh.first_dx_hcp_5yr,
    fdh.first_dx_all_visit_count_5yr,
    fdh.first_dx_last_visit_5yr,

    fth.first_tx_hcp_5yr,
    fth.first_tx_all_visit_count_5yr,
    fth.first_tx_treatment_visit_count_5yr,
    fth.first_tx_last_visit_5yr,

    msh1.NPI AS most_seen_hcp1_3yr_ranked,
    msh1.all_visit_count_5yr AS most_seen_hcp1_visit_count_5yr,
    msh1.last_visit_5yr AS most_seen_hcp1_last_visit_5yr,

    msh2.NPI AS most_seen_hcp2_3yr_ranked,
    msh2.all_visit_count_5yr AS most_seen_hcp2_visit_count_5yr,
    msh2.last_visit_5yr AS most_seen_hcp2_last_visit_5yr,

    msh3.NPI AS most_seen_hcp3_3yr_ranked,
    msh3.all_visit_count_5yr AS most_seen_hcp3_visit_count_5yr,
    msh3.last_visit_5yr AS most_seen_hcp3_last_visit_5yr,

    msh4.NPI AS most_seen_hcp4_3yr_ranked,
    msh4.all_visit_count_5yr AS most_seen_hcp4_visit_count_5yr,
    msh4.last_visit_5yr AS most_seen_hcp4_last_visit_5yr,

    msh5.NPI AS most_seen_hcp5_3yr_ranked,
    msh5.all_visit_count_5yr AS most_seen_hcp5_visit_count_5yr,
    msh5.last_visit_5yr AS most_seen_hcp5_last_visit_5yr,

    ph.PRIMARY_HCP_NPI,
    ph.PRIMARY_HCP_SPECIALTY,
    ph.SPECIALTY_PRIORITY,
    ph.NO_OF_VISITS,
    ph.DX_VISITS,
    ph.TX_VISITS,
    ph.MOST_RECENT_VISIT,
    ph.HCP_RANK,

    ph.primary_hcp_name_2yr,
    ph.primary_hcp_hco_name_2yr,
    ph.primary_hcp_hco_city_2yr,
    ph.primary_hcp_hco_state_2yr,
    ph.primary_hcp_territory_id_2yr,
    ph.primary_hcp_territory_2yr,
    ph.primary_hcp_region_id_2yr,
    ph.primary_hcp_region_2yr,

    ep.FLAG_2DX,
    ep.FLAG_TX_EVER,
    ep.FLAG_2DX_AND_RECENT_TX,
    ep.FLAG_TXEVER_AND_RECENT_TX,
    COALESCE(elf.FLAG_ELAPRASE_LAST_2YR,0) AS FLAG_ELAPRASE_LAST_2YR,
    COALESCE(ocf.OLD_COHORT_FLAG,0) AS OLD_COHORT_FLAG

FROM eligible_patients_new ep

LEFT JOIN old_cohort_flag ocf
    ON ep.PATIENT_ID = ocf.PATIENT_ID

LEFT JOIN patient_demographics pd
    ON ep.PATIENT_ID = pd.PATIENT_ID

LEFT JOIN patient_geography pg
    ON ep.PATIENT_ID = pg.PATIENT_ID

LEFT JOIN incidence_dates idx
    ON ep.PATIENT_ID = idx.PATIENT_ID

LEFT JOIN first_tx_dates ftx
    ON ep.PATIENT_ID = ftx.PATIENT_ID

LEFT JOIN latest_claim_dates lcd
    ON ep.PATIENT_ID = lcd.PATIENT_ID

LEFT JOIN latest_tx_dates ltd
    ON ep.PATIENT_ID = ltd.PATIENT_ID

LEFT JOIN latest_tx_type ltt
    ON ep.PATIENT_ID = ltt.PATIENT_ID

LEFT JOIN elaprase_fills ef
    ON ep.PATIENT_ID = ef.PATIENT_ID

LEFT JOIN elaprase_last_2yr_flag elf
    ON ep.PATIENT_ID = elf.PATIENT_ID

LEFT JOIN latest_claim_hcp lch
    ON ep.PATIENT_ID = lch.PATIENT_ID

LEFT JOIN latest_treatment_hcp lth
    ON ep.PATIENT_ID = lth.PATIENT_ID

LEFT JOIN first_dx_hcp fdh
    ON ep.PATIENT_ID = fdh.PATIENT_ID

LEFT JOIN first_tx_hcp fth
    ON ep.PATIENT_ID = fth.PATIENT_ID

LEFT JOIN most_seen_hcps msh1
    ON ep.PATIENT_ID = msh1.PATIENT_ID
   AND msh1.visit_rank = 1

LEFT JOIN most_seen_hcps msh2
    ON ep.PATIENT_ID = msh2.PATIENT_ID
   AND msh2.visit_rank = 2

LEFT JOIN most_seen_hcps msh3
    ON ep.PATIENT_ID = msh3.PATIENT_ID
   AND msh3.visit_rank = 3

LEFT JOIN most_seen_hcps msh4
    ON ep.PATIENT_ID = msh4.PATIENT_ID
   AND msh4.visit_rank = 4

LEFT JOIN most_seen_hcps msh5
    ON ep.PATIENT_ID = msh5.PATIENT_ID
   AND msh5.visit_rank = 5

LEFT JOIN com_edp_prd.cmpa_insights_internal_schema.primary_hcp_new ph
    ON ep.PATIENT_ID = ph.PATIENT_ID;


In [0]:
SELECT * from com_edp_prd.cmpa_insights_internal_schema.patient360_master_new

In [0]:
describe com_edp_prd.cmpa_insights_internal_schema.patient360_master_new

In [0]:
-- -- Total patients
SELECT COUNT(DISTINCT PATIENT_ID)
FROM eligible_patients_new;

In [0]:
-- Patients from 2 Dx pathway
SELECT COUNT(DISTINCT PATIENT_ID)
FROM eligible_patients_new
WHERE FLAG_2DX_AND_RECENT_TX = 1;

In [0]:
-- Patients from Tx-ever pathway
SELECT COUNT(DISTINCT PATIENT_ID)
FROM eligible_patients_new
WHERE FLAG_TXEVER_AND_RECENT_TX = 1;

In [0]:
-- Overlap patients
SELECT COUNT(DISTINCT PATIENT_ID)
FROM eligible_patients_new
WHERE FLAG_2DX = 1
  AND FLAG_TX_EVER = 1;